# LeetCode 696 — Count Binary Substrings bằng Python

**Link đề:** https://leetcode.com/problems/count-binary-substrings/
**Độ khó:** Easy · **Chủ đề:** Two Pointers, String

```python
class Solution:
    def countBinarySubstrings(self, s: str) -> int:
        ...
```

## Mục tiêu học
1. Hiểu đúng điều kiện "số `0` bằng số `1` **và** các `0`, các `1` đứng **liền thành khối**".
2. Viết được cách brute force (sinh mọi substring rồi kiểm tra) để hiểu bài, thấy vì sao nó chậm.
3. Rút ra insight **chia chuỗi thành các nhóm (run) ký tự giống nhau** và công thức `min(nhóm trước, nhóm sau)`.
4. Cài đặt 4 cách: brute force → danh sách độ dài nhóm → hai biến `prev/cur` O(1) bộ nhớ → `itertools.groupby`.
5. Trace từng bước, chứng minh đúng, phân tích Time/Space, nhận dạng pattern "run-length" cho bài khác.

## 1. Đề bài và ví dụ

Cho chuỗi nhị phân `s` (chỉ gồm `'0'` và `'1'`). Đếm số **substring khác rỗng** thỏa **đồng thời**:
- số ký tự `'0'` bằng số ký tự `'1'`;
- tất cả `'0'` đứng liền nhau và tất cả `'1'` đứng liền nhau (tức substring có dạng `0...01...1` hoặc `1...10...0`).

Substring xuất hiện nhiều lần ở **vị trí khác nhau** thì đếm nhiều lần.

| `s` | Kết quả | Các substring hợp lệ |
|---|:---:|---|
| `"00110011"` | `6` | `"0011"`, `"01"`, `"1100"`, `"10"`, `"0011"`, `"01"` |
| `"10101"` | `4` | `"10"`, `"01"`, `"10"`, `"01"` |

**Constraints:** `1 <= len(s) <= 10^5`, `s[i]` là `'0'` hoặc `'1'`.

Chú ý: `"00110011"` (cả chuỗi) **không** hợp lệ dù có 4 số 0 và 4 số 1, vì các số 0 bị tách thành 2 khối.

In [1]:
s1 = "00110011"
s2 = "10101"
print("s1 =", s1, "| độ dài:", len(s1), "| số '0':", s1.count("0"), "| số '1':", s1.count("1"))
print("s2 =", s2, "| độ dài:", len(s2), "| số '0':", s2.count("0"), "| số '1':", s2.count("1"))

s1 = 00110011 | độ dài: 8 | số '0': 4 | số '1': 4
s2 = 10101 | độ dài: 5 | số '0': 2 | số '1': 3


## 2. Nền tảng Python cần dùng

### 2.1. Indexing và slicing chuỗi

- `s[i]` lấy ký tự ở vị trí `i` (đếm từ 0).
- `s[i:j]` lấy substring từ `i` **đến trước** `j` (không lấy `j`). Độ dài = `j - i`.

Mọi substring của `s` đều có dạng `s[i:j]` với `0 <= i < j <= len(s)`.

In [2]:
s = "00110011"
print(s[0], s[2])        # '0' '1'
print(s[0:4])            # '0011'  (vị trí 0,1,2,3)
print(s[2:4])            # '11'
print(len(s[1:5]))       # 4 = 5 - 1

0 1
0011
11
4


### 2.2. Hai vòng lặp lồng nhau để sinh mọi substring

Với `n = len(s)`, có `n*(n+1)/2` substring. Ví dụ `"011"` có 6 substring.

In [3]:
s = "011"
subs = []
for i in range(len(s)):
    for j in range(i + 1, len(s) + 1):
        subs.append(s[i:j])
print(subs)
print("Số substring:", len(subs), "= n*(n+1)/2 =", len(s) * (len(s) + 1) // 2)

['0', '01', '011', '1', '11', '1']
Số substring: 6 = n*(n+1)/2 = 6


### 2.3. `set(...)` để kiểm tra "toàn một ký tự"

`set("000")` là `{'0'}` có 1 phần tử → chuỗi chỉ gồm một loại ký tự. Mẹo này giúp kiểm tra nhanh một nửa substring.

### 2.4. `min(a, b)`

Trả về số nhỏ hơn. Sẽ là "trái tim" của lời giải tối ưu.

In [4]:
print(set("000"), len(set("000")))
print(set("001"), len(set("001")))
print(min(3, 5), min(2, 2), min(7, 1))

{'0'} 1
{'0', '1'} 2
3 2 1


### 2.5. Run (nhóm liên tiếp) là gì?

**Run** = đoạn dài nhất các ký tự giống nhau đứng liền. Ví dụ `"00110011"` có 4 run: `"00" | "11" | "00" | "11"`, độ dài `[2, 2, 2, 2]`.
`"0001111000"` có run `[3, 4, 3]`. Việc nén chuỗi thành độ dài các run gọi là **run-length encoding**.

In [5]:
def run_lengths(s: str) -> list[int]:
    groups = [1]                    # nhóm đầu tiên chứa s[0]
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            groups[-1] += 1         # cùng ký tự với trước -> kéo dài nhóm cuối
        else:
            groups.append(1)        # đổi ký tự -> mở nhóm mới
    return groups

for t in ["00110011", "10101", "0001111000", "0"]:
    print(f"{t!r:14} -> {run_lengths(t)}")

'00110011'     -> [2, 2, 2, 2]
'10101'        -> [1, 1, 1, 1, 1]
'0001111000'   -> [3, 4, 3]
'0'            -> [1]


Giải thích cú pháp mới:
- `groups[-1]` là phần tử **cuối** của list (`-1` đếm từ cuối).
- `groups[-1] += 1` nghĩa là `groups[-1] = groups[-1] + 1`.
- `groups.append(1)` thêm phần tử `1` vào cuối list.
- `f"{t!r:14}"`: `!r` in kèm dấu nháy, `:14` canh độ rộng 14 ký tự cho dễ đọc.

## 3. Hiểu đúng điều kiện hợp lệ — viết hàm kiểm tra

Substring `t` hợp lệ khi:
1. `len(t)` chẵn, gọi `half = len(t) // 2`;
2. nửa đầu `t[:half]` toàn một ký tự;
3. nửa sau `t[half:]` toàn một ký tự;
4. hai ký tự đó **khác nhau**.

Vì sao chỉ cần chia đôi? Nếu số 0 = số 1 = `half` và mỗi loại đứng thành một khối, thì khối đầu chiếm đúng `half` ký tự đầu, khối sau chiếm `half` ký tự sau.

In [6]:
def is_valid(t: str) -> bool:
    if len(t) % 2 == 1:
        return False
    half = len(t) // 2
    left, right = t[:half], t[half:]
    return len(set(left)) == 1 and len(set(right)) == 1 and left[0] != right[0]

for t in ["01", "10", "0011", "1100", "0101", "00110011", "000111", "0010", "00"]:
    print(f"{t:10} -> {is_valid(t)}")

01         -> True
10         -> True
0011       -> True
1100       -> True
0101       -> False
00110011   -> False
000111     -> True
0010       -> False
00         -> False


## 4. Cách 1 — Brute force: thử mọi substring

Ý tưởng tự nhiên nhất: duyệt mọi cặp `(i, j)`, cắt `s[i:j]`, gọi `is_valid`, đếm.

- Số substring: `O(n²)`.
- Mỗi lần cắt và kiểm tra: `O(n)`.
- Tổng: `O(n³)` — quá chậm cho `n = 10^5` (khoảng `10^15` phép).

In [7]:
def count_brute(s: str) -> int:
    count = 0
    n = len(s)
    for i in range(n):
        for j in range(i + 2, n + 1, 2):    # chỉ thử độ dài chẵn
            if is_valid(s[i:j]):
                count += 1
    return count

print(count_brute("00110011"))   # 6
print(count_brute("10101"))      # 4

6
4


### 4.1. Cải tiến nhỏ — mở rộng từ "ranh giới" (O(n²))

Quan sát: mọi substring hợp lệ đều có **đúng một chỗ đổi ký tự** ở chính giữa. Vậy với mỗi vị trí `i` mà `s[i] != s[i+1]` (ranh giới), ta mở rộng sang hai bên chừng nào bên trái vẫn bằng `s[i]` và bên phải vẫn bằng `s[i+1]`. Mỗi bước mở rộng thành công là một substring hợp lệ.

Đây giống kỹ thuật "expand around center" trong bài palindrome.

In [8]:
def count_expand(s: str) -> int:
    count = 0
    n = len(s)
    for i in range(n - 1):
        if s[i] == s[i + 1]:
            continue                  # không phải ranh giới
        left, right = i, i + 1
        while left >= 0 and right < n and s[left] == s[i] and s[right] == s[i + 1]:
            count += 1                # s[left:right+1] hợp lệ
            left -= 1
            right += 1
    return count

print(count_expand("00110011"), count_expand("10101"), count_expand("0001111000"))

6 4 6


Cách này đúng nhưng trường hợp xấu vẫn `O(n²)`? Thật ra mỗi ranh giới chỉ mở rộng tối đa `min(run trái, run phải)` bước, và tổng các `min` không vượt `n`, nên `count_expand` thực tế là `O(n)`. Tuy vậy, chính phép đếm "số bước mở rộng = min(run trái, run phải)" mới là insight quan trọng — ta sẽ làm rõ ở mục sau.

## 5. Nút thắt của brute force

Brute force lặp lại cùng một việc: kiểm tra đi kiểm tra lại các đoạn ký tự giống nhau. Ví dụ với `"000111"` nó kiểm tra `"01"`, `"0011"`, `"000111"` riêng rẽ, mỗi lần đều quét lại các ký tự.

Câu hỏi then chốt (theo hint của đề): **`"000111"` có bao nhiêu substring hợp lệ? `"11100"` thì sao? `"00011100"`?**

In [9]:
for t in ["000111", "11100", "00011100", "0011100"]:
    print(f"{t:10} runs={run_lengths(t)} -> brute = {count_brute(t)}")

000111     runs=[3, 3] -> brute = 3
11100      runs=[3, 2] -> brute = 2
00011100   runs=[3, 3, 2] -> brute = 5
0011100    runs=[2, 3, 2] -> brute = 4


## 6. Insight — mỗi cặp nhóm kề nhau đóng góp `min(a, b)`

Nhìn kết quả:
- `"000111"`: run `[3, 3]` → `3` substring: `"01"`, `"0011"`, `"000111"`.
- `"11100"`: run `[3, 2]` → `2` substring: `"10"`, `"1100"`. Không thể lấy `"111000"` vì chỉ có 2 số 0.
- `"00011100"`: run `[3, 3, 2]` → `min(3,3) + min(3,2) = 3 + 2 = 5`.

**Quy tắc:** substring hợp lệ luôn nằm vắt qua **đúng một ranh giới** giữa hai run kề nhau có độ dài `a` và `b`. Nó lấy `k` ký tự cuối của run trái và `k` ký tự đầu của run phải với `1 <= k <= min(a, b)`. Vậy:

```text
đáp án = tổng min(groups[i-1], groups[i])  với mọi i từ 1 đến len(groups)-1
```

In [10]:
a, b_ = 3, 2
print("run trái dài", a, ", run phải dài", b_, "-> số substring:", min(a, b_))
for k in range(1, min(a, b_) + 1):
    print("  k =", k, "->", "1" * k + "0" * k)

run trái dài 3 , run phải dài 2 -> số substring: 2
  k = 1 -> 10
  k = 2 -> 1100


## 7. Cách 2 — Danh sách độ dài nhóm (O(n) thời gian, O(n) bộ nhớ)

Ghép hai bước:
1. `groups = run_lengths(s)`.
2. Cộng `min` của mọi cặp kề nhau.

Ở bước 2 dùng `zip(groups, groups[1:])` để duyệt cặp kề nhau: `zip` ghép phần tử cùng vị trí của hai dãy.

In [11]:
groups = [3, 3, 2]
print(groups[1:])                       # [3, 2]
print(list(zip(groups, groups[1:])))    # [(3, 3), (3, 2)]

[3, 2]
[(3, 3), (3, 2)]


In [12]:
def count_groups(s: str) -> int:
    groups = run_lengths(s)
    total = 0
    for a, c in zip(groups, groups[1:]):
        total += min(a, c)
    return total

for t in ["00110011", "10101", "00011100", "0"]:
    print(f"{t:10} -> {count_groups(t)}")

00110011   -> 6
10101      -> 4
00011100   -> 5
0          -> 0


## 8. Cách 3 (lời giải chính) — chỉ giữ hai biến `prev` và `cur` (O(1) bộ nhớ)

Ta không cần lưu cả list `groups`: tại mỗi thời điểm chỉ cần **độ dài run trước** (`prev`) và **độ dài run đang đếm** (`cur`).

Mổ xẻ từng đoạn:
- Khởi tạo `prev = 0` (chưa có run trước), `cur = 1` (run hiện tại chứa `s[0]`), `ans = 0`.
- Duyệt `i` từ 1: nếu `s[i] == s[i-1]` → `cur += 1`.
- Nếu đổi ký tự → run hiện tại kết thúc: cộng `min(prev, cur)` vào `ans`, rồi dịch: `prev = cur`, `cur = 1`.
- Sau vòng lặp, run cuối chưa được "chốt" → cộng thêm `min(prev, cur)`.

Vì sao `prev = 0` ban đầu? Để run đầu tiên khi chốt cộng `min(0, cur) = 0` — run đầu không có bạn bên trái.

In [13]:
def count_two_vars(s: str) -> int:
    prev, cur, ans = 0, 1, 0
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            cur += 1
        else:
            ans += min(prev, cur)
            prev, cur = cur, 1
    return ans + min(prev, cur)

for t in ["00110011", "10101", "00011100", "0", "01"]:
    print(f"{t:10} -> {count_two_vars(t)}")

00110011   -> 6
10101      -> 4
00011100   -> 5
0          -> 0
01         -> 1


**Cú pháp `prev, cur = cur, 1`**: gán đồng thời (tuple unpacking). Vế phải được tính xong **trước** rồi mới gán, nên `prev` nhận giá trị **cũ** của `cur`. Nếu viết tách `prev = cur; cur = 1` cũng đúng ở đây, nhưng gán đồng thời giúp tránh lỗi khi hai biến phụ thuộc lẫn nhau.

In [14]:
prev, cur = 2, 5
prev, cur = cur, 1
print(prev, cur)   # 5 1

5 1


## 9. TRACE từng bước — `s = "00110011"` và `s = "0001100"`

Bảng trace cho `"0001100"` (run `[3, 2, 2]`, đáp án `min(3,2) + min(2,2) = 4`):

| i | s[i-1] → s[i] | Đổi ký tự? | Cộng `min(prev,cur)` | prev | cur | ans |
|---|---|:---:|---|:---:|:---:|:---:|
| bắt đầu | | | | 0 | 1 | 0 |
| 1 | 0 → 0 | không | | 0 | 2 | 0 |
| 2 | 0 → 0 | không | | 0 | 3 | 0 |
| 3 | 0 → 1 | có | min(0,3)=0 | 3 | 1 | 0 |
| 4 | 1 → 1 | không | | 3 | 2 | 0 |
| 5 | 1 → 0 | có | min(3,2)=2 | 2 | 1 | 2 |
| 6 | 0 → 0 | không | | 2 | 2 | 2 |
| kết thúc | | chốt run cuối | min(2,2)=2 | | | **4** |

Hàm trace in đúng các cột này.

In [15]:
def trace_count(s: str) -> int:
    prev, cur, ans = 0, 1, 0
    print(f"start: prev={prev}, cur={cur}, ans={ans}")
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            cur += 1
            print(f"i={i}: {s[i-1]}->{s[i]} giống  | prev={prev}, cur={cur}, ans={ans}")
        else:
            add = min(prev, cur)
            ans += add
            prev, cur = cur, 1
            print(f"i={i}: {s[i-1]}->{s[i]} ĐỔI   | +{add} | prev={prev}, cur={cur}, ans={ans}")
    add = min(prev, cur)
    print(f"chốt run cuối: +min({prev},{cur}) = +{add} -> ans = {ans + add}")
    return ans + add

trace_count("0001100")

start: prev=0, cur=1, ans=0
i=1: 0->0 giống  | prev=0, cur=2, ans=0
i=2: 0->0 giống  | prev=0, cur=3, ans=0
i=3: 0->1 ĐỔI   | +0 | prev=3, cur=1, ans=0
i=4: 1->1 giống  | prev=3, cur=2, ans=0
i=5: 1->0 ĐỔI   | +2 | prev=2, cur=1, ans=2
i=6: 0->0 giống  | prev=2, cur=2, ans=2
chốt run cuối: +min(2,2) = +2 -> ans = 4


4

In [16]:
trace_count("00110011")

start: prev=0, cur=1, ans=0
i=1: 0->0 giống  | prev=0, cur=2, ans=0
i=2: 0->1 ĐỔI   | +0 | prev=2, cur=1, ans=0
i=3: 1->1 giống  | prev=2, cur=2, ans=0
i=4: 1->0 ĐỔI   | +2 | prev=2, cur=1, ans=2
i=5: 0->0 giống  | prev=2, cur=2, ans=2
i=6: 0->1 ĐỔI   | +2 | prev=2, cur=1, ans=4
i=7: 1->1 giống  | prev=2, cur=2, ans=4
chốt run cuối: +min(2,2) = +2 -> ans = 6


6

## 10. Cách 4 — `itertools.groupby` (ngắn gọn kiểu Python)

`itertools.groupby(s)` gom các phần tử **liền nhau** giống nhau thành nhóm, trả về cặp `(key, iterator)`. `len(list(g))` cho độ dài nhóm.

In [17]:
from itertools import groupby

for key, g in groupby("0001100"):
    print(key, list(g))

0 ['0', '0', '0']
1 ['1', '1']
0 ['0', '0']


In [18]:
def count_groupby(s: str) -> int:
    groups = [len(list(g)) for _, g in groupby(s)]
    return sum(map(min, groups, groups[1:]))

print(count_groupby("00110011"), count_groupby("10101"))

6 4


Giải thích:
- `[len(list(g)) for _, g in groupby(s)]` là **list comprehension**: tạo list bằng một dòng; `_` là tên biến "không dùng tới" (key).
- `map(min, groups, groups[1:])` gọi `min(groups[i], groups[i+1])` cho từng cặp — tương đương `zip` + `min`.
- `sum(...)` cộng tất cả.

## 11. Code nộp LeetCode

Bản chính dùng hai biến (O(1) bộ nhớ):

In [19]:
class Solution:
    def countBinarySubstrings(self, s: str) -> int:
        prev, cur, ans = 0, 1, 0
        for i in range(1, len(s)):
            if s[i] == s[i - 1]:
                cur += 1
            else:
                ans += min(prev, cur)
                prev, cur = cur, 1
        return ans + min(prev, cur)

sol = Solution()
print(sol.countBinarySubstrings("00110011"))  # 6
print(sol.countBinarySubstrings("10101"))     # 4

6
4


Bản ngắn dùng `groupby` (cũng được chấp nhận):

```python
from itertools import groupby

class Solution:
    def countBinarySubstrings(self, s: str) -> int:
        groups = [len(list(g)) for _, g in groupby(s)]
        return sum(map(min, groups, groups[1:]))
```

## 12. Vì sao thuật toán đúng?

**Bổ đề 1 — mỗi substring hợp lệ vắt qua đúng một ranh giới run.** Substring hợp lệ có dạng `c^k d^k` (k ký tự `c` rồi k ký tự `d`, `c != d`). Nó chứa đúng một chỗ đổi ký tự, chính là một ranh giới giữa run `A` (chứa các `c`) và run `B` (chứa các `d`) kề nhau. Các `c` phải là `k` ký tự **cuối** của `A`, các `d` là `k` ký tự **đầu** của `B` (nếu không, substring sẽ chứa thêm ranh giới khác).

**Bổ đề 2 — với ranh giới giữa run dài `a` và `b`, số substring hợp lệ đúng bằng `min(a, b)`.** Mỗi `k` từ `1` đến `min(a,b)` cho đúng một substring; `k > min(a,b)` thì một bên thiếu ký tự.

Hai bổ đề cho: các substring được chia thành các nhóm rời nhau theo ranh giới → không đếm trùng, không bỏ sót → đáp án = tổng `min`.

**Invariant của vòng lặp hai biến:** sau khi xử lý `s[0..i]`, `cur` = độ dài run chứa `s[i]` (tính tới `i`), `prev` = độ dài run ngay trước nó (0 nếu chưa có), `ans` = tổng `min` của mọi cặp run kề nhau **đã kết thúc hoàn toàn**. Bước chốt cuối thêm cặp (run áp chót, run cuối).

## 13. Độ phức tạp và so sánh các cách

Gọi `n = len(s)`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | Thử mọi substring độ dài chẵn | `O(n³)` | `O(n)` (substring tạm) | Chỉ để hiểu đề |
| 1b. Mở rộng từ ranh giới | Expand around boundary | `O(n)` (tổng min ≤ n) | `O(1)` | Trực quan, cầu nối tới insight |
| 2. List độ dài run | `sum(min(cặp kề))` | `O(n)` | `O(n)` | Dễ đọc |
| 3. Hai biến `prev/cur` | Như 2 nhưng không lưu list | `O(n)` | `O(1)` | **Nên nộp** |
| 4. `groupby` | Như 2, viết gọn | `O(n)` | `O(n)` | Pythonic |

Vì sao cách 1b là `O(n)`: mỗi ranh giới mở rộng `min(a, b)` bước, và `sum(min(a_i, a_{i+1})) <= sum(a_i) = n`.

## 14. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu nhiều cách

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:28} PASS ({actual})")

sol = Solution()
check("mẫu 1", sol.countBinarySubstrings("00110011"), 6)
check("mẫu 2", sol.countBinarySubstrings("10101"), 4)
check("một ký tự", sol.countBinarySubstrings("0"), 0)
check("toàn 1", sol.countBinarySubstrings("1111"), 0)
check("01", sol.countBinarySubstrings("01"), 1)
check("run lệch [3,1]", sol.countBinarySubstrings("0001"), 1)
check("run [3,3,2]", sol.countBinarySubstrings("00011100"), 5)
check("phản ví dụ: cả chuỗi 4-4", sol.countBinarySubstrings("00110011"), 6)

mẫu 1                        PASS (6)
mẫu 2                        PASS (4)
một ký tự                    PASS (0)
toàn 1                       PASS (0)
01                           PASS (1)
run lệch [3,1]               PASS (1)
run [3,3,2]                  PASS (5)
phản ví dụ: cả chuỗi 4-4     PASS (6)


### 14.1. Đối chiếu tất cả cách trên mọi chuỗi nhị phân độ dài 1..10 và trên chuỗi ngẫu nhiên

`itertools.product("01", repeat=L)` sinh mọi chuỗi độ dài `L` gồm `0/1`. Brute force làm **oracle** (đáp án chắc chắn đúng) để kiểm các cách nhanh.

In [21]:
from itertools import product
import random

checked = 0
for L in range(1, 11):
    for tup in product("01", repeat=L):
        t = "".join(tup)
        expected = count_brute(t)
        assert count_expand(t) == expected
        assert count_groups(t) == expected
        assert count_two_vars(t) == expected
        assert count_groupby(t) == expected
        checked += 1
print("Đã đối chiếu", checked, "chuỗi: tất cả khớp")

rng = random.Random(696)
for _ in range(200):
    t = "".join(rng.choice("01") for _ in range(rng.randint(1, 40)))
    assert count_two_vars(t) == count_brute(t) == count_groupby(t)
print("200 chuỗi ngẫu nhiên: PASS")

Đã đối chiếu 2046 chuỗi: tất cả khớp
200 chuỗi ngẫu nhiên: PASS


### 14.2. Test lớn — kiểm tra tốc độ

Chuỗi xen kẽ `"0101..."` độ dài `10^5` có `n - 1` ranh giới, mỗi ranh giới đóng góp 1 → đáp án `99999`.

In [22]:
import time
big = "01" * 50000
t0 = time.perf_counter()
res = Solution().countBinarySubstrings(big)
print(res, "| thời gian:", round(time.perf_counter() - t0, 4), "giây")
assert res == 99999
big2 = "0" * 50000 + "1" * 50000
assert Solution().countBinarySubstrings(big2) == 50000
print("Test lớn PASS")

99999 | thời gian: 0.0162 giây


Test lớn PASS


## 15. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm số 0 bằng số 1, quên điều kiện "liền khối"

Sai: đếm `"0101"`, `"00110011"` là hợp lệ.

In [23]:
def wrong_only_balance(s: str) -> int:
    count = 0
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            t = s[i:j]
            if t.count("0") == t.count("1"):
                count += 1
    return count

print("sai:", wrong_only_balance("00110011"), "| đúng:", count_brute("00110011"))

sai: 10 | đúng: 6


Vì sao sai: `"0101"`, `"1001"`, `"00110011"`... cân bằng nhưng không liền khối. Sửa: kiểm tra đúng dạng `c^k d^k` (hàm `is_valid`) hoặc dùng insight run.

### Lỗi 2 — Quên chốt run cuối sau vòng lặp

In [24]:
def wrong_forget_last(s: str) -> int:
    prev, cur, ans = 0, 1, 0
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            cur += 1
        else:
            ans += min(prev, cur)
            prev, cur = cur, 1
    return ans          # thiếu + min(prev, cur)

print("sai:", wrong_forget_last("00110011"), "| đúng:", count_two_vars("00110011"))

sai: 4 | đúng: 6


Vì sao sai: cặp (run áp chót, run cuối) chỉ được cộng khi gặp ranh giới **tiếp theo**, mà run cuối không có ranh giới sau. Sửa: `return ans + min(prev, cur)`.

### Lỗi 3 — Cộng `max` hoặc `a + b` thay vì `min`

`"11100"` có run `[3, 2]`; nếu cộng `max` sẽ ra 3 (đếm cả `"111000"` không tồn tại). Phải là `min`: số cặp bị giới hạn bởi bên ít hơn.

### Lỗi 4 — Khởi tạo `cur = 0` và duyệt từ `i = 0`

Khi `i = 0`, `s[i - 1]` là `s[-1]` (ký tự **cuối** chuỗi trong Python!), không báo lỗi mà cho kết quả sai lặng lẽ.

In [25]:
s = "0110"
print("s[-1] =", s[-1], "-> so sánh s[0] với s[-1] là so với ký tự cuối, không phải 'trước' s[0]")

s[-1] = 0 -> so sánh s[0] với s[-1] là so với ký tự cuối, không phải 'trước' s[0]


## 16. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu dùng run-length / nhóm liên tiếp:**
- Đề nói "liên tiếp", "consecutive", "grouped together", "khối".
- Kết quả chỉ phụ thuộc **độ dài** các đoạn ký tự giống nhau, không phụ thuộc vị trí tuyệt đối.
- Bài liên quan: 485 Max Consecutive Ones, 1446 Consecutive Characters, 443 String Compression, 1869 Longer Contiguous Segments of Ones than Zeros.

**Template run-length bằng hai biến:**
```python
cur = 1
for i in range(1, len(s)):
    if s[i] == s[i-1]:
        cur += 1
    else:
        # xử lý run vừa kết thúc (độ dài cur)
        cur = 1
# xử lý run cuối (độ dài cur)
```

### 16.1. Biến thể A — liệt kê các substring hợp lệ (kèm vị trí)

In [26]:
def list_valid(s: str) -> list[tuple[int, str]]:
    out = []
    for i in range(len(s) - 1):
        if s[i] != s[i + 1]:
            l, r = i, i + 1
            while l >= 0 and r < len(s) and s[l] == s[i] and s[r] == s[i + 1]:
                out.append((l, s[l:r + 1]))
                l -= 1
                r += 1
    return out

print(list_valid("00110011"))

[(1, '01'), (0, '0011'), (3, '10'), (2, '1100'), (5, '01'), (4, '0011')]


### 16.2. Biến thể B — substring hợp lệ dài nhất

Mỗi ranh giới cho substring dài nhất `2 * min(a, b)` → lấy `max`.

In [27]:
def longest_valid(s: str) -> int:
    groups = run_lengths(s)
    return max((2 * min(a, c) for a, c in zip(groups, groups[1:])), default=0)

print(longest_valid("00110011"), longest_valid("0001110"), longest_valid("0000"))

4 6 0


`max(..., default=0)` trả `0` khi dãy rỗng (chuỗi chỉ có một run), tránh lỗi `ValueError`.

## 17. Bài tập tự luyện (tăng dần)

**Bài 1 (tính tay).** Tính run và đáp án cho `s = "1100011"`. Viết kết quả vào biến `ex1_answer`.

**Bài 2 (TODO).** Viết `ex2_run_lengths(s)` trả về list độ dài run (không dùng `groupby`).

**Bài 3 (TODO).** Viết lại lời giải hai biến `ex3_count(s)` mà không nhìn code phía trên.

**Bài 4 (biến thể).** `ex4_count_only_zero_first(s)`: chỉ đếm substring hợp lệ dạng `0...01...1` (số 0 đứng trước).

**Bài 5 (biến thể).** `ex5_max_consecutive(s, ch)`: độ dài run dài nhất của ký tự `ch` (LeetCode 485 tổng quát).

**Bài 6 (mini challenge).** `ex6_count_k(s, k)`: đếm substring hợp lệ có **đúng** `k` số 0 và `k` số 1 (độ dài `2k`). Gợi ý: một ranh giới (a, b) đóng góp 1 nếu `min(a,b) >= k`.

In [28]:
# Bài 1: điền đáp án tính tay
ex1_answer = None   # TODO

def ex2_run_lengths(s: str) -> list[int]:
    # TODO: duyệt từ i = 1, kéo dài nhóm cuối hoặc mở nhóm mới
    pass

def ex3_count(s: str) -> int:
    # TODO: prev, cur, ans; nhớ chốt run cuối
    pass

# Tự làm xong rồi mới mở test:
# assert ex1_answer == ...
# assert ex2_run_lengths("0001100") == [3, 2, 2]
# assert ex3_count("00110011") == 6

In [29]:
def ex4_count_only_zero_first(s: str) -> int:
    # TODO: chỉ cộng min(prev, cur) khi run trước là '0'
    pass

def ex5_max_consecutive(s: str, ch: str) -> int:
    # TODO
    pass

def ex6_count_k(s: str, k: int) -> int:
    # TODO
    pass

# assert ex4_count_only_zero_first("00110011") == 4
# assert ex5_max_consecutive("1101110", "1") == 3
# assert ex6_count_k("00110011", 2) == 3

## 18. Gợi ý 3 mức, đáp án và tổng kết

### Gợi ý
- **Mức 1 (hướng nghĩ):** mọi thứ quy về danh sách độ dài run và các cặp run kề nhau.
- **Mức 2 (cấu trúc):** Bài 4 cần biết ký tự của run trước: đó là `s[i-1]` tại thời điểm đổi ký tự. Bài 6: ranh giới (a,b) chứa substring dạng `c^k d^k` khi và chỉ khi `k <= min(a,b)`.
- **Mức 3 (gần code):** Bài 4 — trong nhánh `else`, nếu `s[i - 1] == '1'` thì run vừa kết thúc là `'1'`, run trước nó là `'0'` → cộng `min(prev, cur)`. Lúc chốt cuối, kiểm tra `s[-1] == '1'`.

### Đáp án tham khảo (chạy kiểm tra)
Bài 1: `"1100011"` → run `[2, 3, 2]` → `min(2,3) + min(3,2) = 2 + 2 = 4`.

In [30]:
ans_ex1 = 4

def ans_ex2_run_lengths(s: str) -> list[int]:
    groups = [1]
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            groups[-1] += 1
        else:
            groups.append(1)
    return groups

def ans_ex3_count(s: str) -> int:
    prev, cur, ans = 0, 1, 0
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            cur += 1
        else:
            ans += min(prev, cur)
            prev, cur = cur, 1
    return ans + min(prev, cur)

def ans_ex4_count_only_zero_first(s: str) -> int:
    prev, cur, ans = 0, 1, 0
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            cur += 1
        else:
            if s[i - 1] == "1":          # run vừa xong là '1' => run trước là '0'
                ans += min(prev, cur)
            prev, cur = cur, 1
    if s[-1] == "1":
        ans += min(prev, cur)
    return ans

def ans_ex5_max_consecutive(s: str, ch: str) -> int:
    best = cur = 0
    for c in s:
        cur = cur + 1 if c == ch else 0
        best = max(best, cur)
    return best

def ans_ex6_count_k(s: str, k: int) -> int:
    g = ans_ex2_run_lengths(s)
    return sum(1 for a, c in zip(g, g[1:]) if min(a, c) >= k)

assert ans_ex1 == count_brute("1100011")
assert ans_ex2_run_lengths("0001100") == [3, 2, 2]
assert ans_ex3_count("00110011") == 6
assert ans_ex4_count_only_zero_first("00110011") == 4
assert ans_ex5_max_consecutive("1101110", "1") == 3
assert ans_ex6_count_k("00110011", 2) == 3
for L in range(1, 9):
    for tup in product("01", repeat=L):
        t = "".join(tup)
        assert ans_ex3_count(t) == count_brute(t)
        brute_zero_first = sum(1 for i in range(L) for j in range(i + 2, L + 1, 2)
                               if is_valid(t[i:j]) and t[i] == "0")
        assert ans_ex4_count_only_zero_first(t) == brute_zero_first
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


### Template nhớ nhanh

```python
class Solution:
    def countBinarySubstrings(self, s: str) -> int:
        prev, cur, ans = 0, 1, 0
        for i in range(1, len(s)):
            if s[i] == s[i - 1]:
                cur += 1
            else:
                ans += min(prev, cur)
                prev, cur = cur, 1
        return ans + min(prev, cur)
```

Câu thần chú: **"Nén thành run → mỗi cặp run kề nhau cho `min(a, b)`."**

### Checklist trước khi submit
- [ ] Đã hiểu substring phải có dạng `0^k1^k` hoặc `1^k0^k` (không chỉ cân bằng).
- [ ] Khởi tạo `prev = 0`, `cur = 1`, duyệt từ `i = 1`.
- [ ] Dùng `min`, không dùng `max` hay `+`.
- [ ] Đã cộng `min(prev, cur)` cho run cuối sau vòng lặp.
- [ ] Test `"0"`, `"01"`, `"0001"`, chuỗi toàn một ký tự.
- [ ] Độ phức tạp `O(n)` thời gian, `O(1)` bộ nhớ.